# Share-Forge - Train on Colab (T4 GPU)

End-to-end training of the three-stage ML pipeline for the Tata Gold ETF RL trading agent.

1. **LSTM Forecaster** - PyTorch + Gaussian NLL
2. **Behavior Cloning policy** - supervised cross-entropy from a perfect-hindsight expert
3. **PPO + LSTM** - RecurrentPPO with the forecaster's signal injected into the obs space

**Hard data cutoff**: model sees only TATAGOLD.NS bars dated `<= 2026-03-31`. Bars on or after `2026-04-01` are kept on a separate route and never fed to the model.

**Runtime > Change runtime type > T4 GPU** before running.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print(f"torch={torch.__version__}  cuda_available={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

## 2. Get the code

Pick **one** option below.

### Option A - Clone from GitHub (preferred)
Push your local Share-Forge to GitHub once, then clone here:

In [ ]:
# !git clone https://github.com/YOUR-USERNAME/Share-Forge.git
# %cd Share-Forge

### Option B - Upload a zip

On your Mac, run from the parent of `Share-Forge/`:
```bash
zip -r Share-Forge.zip Share-Forge -x '*/__pycache__/*' '*/.venv/*' '*/runs/*' '*/wandb/*' '*/checkpoints/*'
```
Then run this cell and pick the zip from your machine:

In [ ]:
from google.colab import files
uploaded = files.upload()
import os, zipfile
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('.')
%cd Share-Forge
!ls

## 3. Install dependencies

Colab already ships PyTorch with CUDA. Install only what's missing.

In [ ]:
!pip install -q yfinance gymnasium 'stable-baselines3>=2.3.0' 'sb3-contrib>=2.3.0' \
    fastapi uvicorn pydantic gradio plotly tensorboard \
    sqlalchemy 'openenv-core>=0.2.0'
import os, sys
sys.path.insert(0, os.getcwd())
print('PYTHONPATH set to', os.getcwd())

## 4. Fetch + cache TATAGOLD.NS

Applies the 2026-03-31 cutoff and saves to `server/data/`.

In [ ]:
!python -m server.data_loader

## 5. Launch TensorBoard

Both training scripts log to `runs/`. The dashboard updates live.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 6. Stage 1 - Train the LSTM forecaster

~1 min on T4. Saves `checkpoints/forecaster.pth` + `checkpoints/forecaster_stats.npz`.

In [ ]:
!python train_forecaster.py \
    --epochs 50 \
    --batch-size 64 \
    --horizon 5 \
    --window-size 20 \
    --hidden-size 64 \
    --num-layers 2 \
    --device cuda

## 7. Stage 2 - Train the BC policy

~30 s on T4. Saves `checkpoints/bc_policy.pth`.

In [ ]:
!python train_bc.py \
    --epochs 30 \
    --batch-size 64 \
    --horizon 5 \
    --buy-threshold 0.005 \
    --sell-threshold -0.005 \
    --device cuda

## 8. Stage 3 - PPO RL fine-tune

~10-15 min on T4 for 200k steps. Uses the forecaster's prediction as an extra observation channel.
Saves `checkpoints/ppo_share_forge.zip`.

In [ ]:
!python train.py \
    --task easy_long_only \
    --timesteps 200000 \
    --device cuda \
    --use-ml-forecaster

## 9. Quick sanity check

Run the trained agent on a fresh backtest and inspect the equity curve.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from server.data_loader import load, slice_by_dates
from server.trading_env import ShareForgeTradingEnv, TradingConfig
from server.policy_loader import predict, policy_status
from server.tasks import get_task
from models import TaskDifficulty

print('Policy status:', policy_status())

spec = get_task(TaskDifficulty.EASY_LONG_ONLY)
df = slice_by_dates(load(), spec.start, spec.end).reset_index(drop=True)
env = ShareForgeTradingEnv(df, TradingConfig(use_ml_forecaster=True))
obs, _ = env.reset()
done = False
while not done:
    raw_window = obs[:, :-2 - (1 if env._forecast_series is not None else 0)].tolist()
    action, _ = predict(raw_window, is_long=env._is_long)
    obs, _, term, trunc, _ = env.step(int(action))
    done = term or trunc

equity = env.equity_curve
prices = df['close'].to_numpy()
bh = (env.config.initial_cash * (prices / prices[0]))[: len(equity)]

plt.figure(figsize=(12, 5))
plt.plot(equity, label='Agent')
plt.plot(bh, label='Buy & Hold', linestyle='--')
plt.legend(); plt.title(f'Backtest: {spec.task_type.value}'); plt.grid(alpha=0.3); plt.show()
print(f'Final equity: {equity[-1]:,.0f}  (B&H: {bh[-1]:,.0f})  trades: {env._n_trades}')

## 10. Bundle and download checkpoints

Downloads the trained models so you can drop them into your local repo's `checkpoints/`.

In [ ]:
!ls -la checkpoints/
!zip -r checkpoints.zip checkpoints/
from google.colab import files
files.download('checkpoints.zip')

## 11. (Optional) Push directly to a Hugging Face Space

Skip the local download and push checkpoints straight into your HF Space repo.

In [ ]:
# !pip install -q huggingface_hub
# from huggingface_hub import HfApi, login
# login(token='hf_YOUR_TOKEN')
# api = HfApi()
# api.upload_folder(
#     folder_path='checkpoints',
#     path_in_repo='checkpoints',
#     repo_id='YOUR-USER/share-forge',
#     repo_type='space',
# )